# Veritas Face: private MobileNetV3 fine-tuning

This is a **free Kaggle** notebook for fine-tuning a pretrained `MobileNetV3-Small` binary portrait classifier. It deliberately trains only from private, reviewable inputs and produces private model artifacts. It is not an authenticity decision service and it does not test, calibrate, or choose a product threshold.

Before running, attach three versioned Kaggle inputs:

1. A read-only snapshot of this repository, named for its exact Git revision.
2. A **private** record dataset containing `portrait-records-v1.json` plus its image files. Never publish it: it contains portrait paths and records.
3. A read-only copy of the official `MobileNet_V3_Small_Weights.IMAGENET1K_V1` checkpoint.

Set the notebook accelerator as desired, keep the private inputs private, and turn Internet access off after the three immutable inputs are attached. The four `REPLACE_…` values below must be pinned before a run starts.

In [ ]:
from pathlib import Path

CONFIG = {
    # An immutable Kaggle Dataset made from this repository at this exact commit.
    'code_root': Path('/kaggle/input/veritas-face-source'),
    'source_revision': 'REPLACE_WITH_40_CHARACTER_GIT_REVISION',
    # A private Kaggle Dataset: JSON manifest plus images relative to data_root.
    'data_root': Path('/kaggle/input/veritas-face-private-records'),
    'record_manifest_name': 'portrait-records-v1.json',
    'record_manifest_sha256': 'REPLACE_WITH_CANONICAL_SHA256_OF_PRIVATE_RECORD_MANIFEST',
    # Download this once from the official torchvision weight URL, hash it, then attach it.
    'weights_path': Path('/kaggle/input/veritas-face-mobilenet-v3-small/mobilenet_v3_small-047dcff4.pth'),
    'weights_sha256': 'REPLACE_WITH_SHA256_OF_OFFICIAL_WEIGHT_FILE',
    'seed': 20260922,
    'image_size': 224,
    'epochs': 8,
    'batch_size': 64,
    'learning_rate': 1e-3,
    'weight_decay': 1e-4,
    'num_workers': 2,
}

for key in ('source_revision', 'record_manifest_sha256', 'weights_sha256'):
    if str(CONFIG[key]).startswith('REPLACE_'):
        raise ValueError(f'Pin CONFIG[{key!r}] to an immutable value before training.')

RECORD_MANIFEST_PATH = CONFIG['data_root'] / CONFIG['record_manifest_name']
OUTPUT_ROOT = Path('/kaggle/working/veritas-face-fine-tune')
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

## Verify the code and every private input before importing PyTorch

`training/fine_tune.py` checks that source and split manifests are exact, labels agree with their catalogue source, real-person subject/capture groups do not cross splits, synthetic generators match their pinned revision, and every attached image matches its recorded SHA-256. A failure is a stop condition; do not waive it.

In [ ]:
import hashlib
import sys

if not (CONFIG['code_root'] / 'training' / 'fine_tune.py').is_file():
    raise FileNotFoundError('Attach an immutable Veritas Face source snapshot at CONFIG[code_root].')
if not CONFIG['weights_path'].is_file():
    raise FileNotFoundError('Attach the pinned MobileNetV3-Small weight file at CONFIG[weights_path].')

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(1024 * 1024), b''):
            digest.update(chunk)
    return f'sha256:{digest.hexdigest()}'

if sha256_file(CONFIG['weights_path']) != CONFIG['weights_sha256']:
    raise ValueError('The attached pretrained checkpoint does not match CONFIG[weights_sha256].')

sys.path.insert(0, str(CONFIG['code_root'] / 'training'))
from fine_tune import (
    LABEL_TO_TARGET,
    binary_auroc,
    build_run_metadata,
    load_record_manifest,
    validate_record_manifest,
    verify_record_files,
    write_json,
)
from validate_manifests import load_json

source_document = load_json(CONFIG['code_root'] / 'training/manifests/portrait-sources-v1.json')
split_document = load_json(CONFIG['code_root'] / 'training/manifests/portrait-splits-v1.json')
record_document = load_record_manifest(RECORD_MANIFEST_PATH)
record_summary = validate_record_manifest(record_document, source_document, split_document)
if record_summary.record_manifest_sha256 != CONFIG['record_manifest_sha256']:
    raise ValueError('The private record manifest does not match CONFIG[record_manifest_sha256].')
verify_record_files(record_document, CONFIG['data_root'])
print(f'Validated {record_summary.record_count} records: {record_summary.records_per_split}')

## Deterministic runtime and auditable configuration

Kaggle image and driver versions can change. The notebook captures package versions and uses deterministic PyTorch operations. If the selected device has no deterministic implementation for an operation, the run must fail rather than silently weaken reproducibility.

In [ ]:
import importlib.metadata
import json
import os
import random

os.environ['CUBLAS_WORKSPACE_CONFIG'] = ':4096:8'
import numpy as np
import torch
import torchvision

random.seed(CONFIG['seed'])
np.random.seed(CONFIG['seed'])
torch.manual_seed(CONFIG['seed'])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG['seed'])
torch.use_deterministic_algorithms(True)
torch.backends.cudnn.benchmark = False
torch.backends.cudnn.deterministic = True

package_versions = {
    'python': sys.version.split()[0],
    'torch': torch.__version__,
    'torchvision': torchvision.__version__,
    'numpy': np.__version__,
    'pillow': importlib.metadata.version('Pillow'),
}
run_metadata = build_run_metadata(
    record_summary=record_summary,
    source_revision=CONFIG['source_revision'],
    seed=CONFIG['seed'],
    image_size=CONFIG['image_size'],
    epochs=CONFIG['epochs'],
    batch_size=CONFIG['batch_size'],
    learning_rate=CONFIG['learning_rate'],
    weight_decay=CONFIG['weight_decay'],
    package_versions=package_versions,
)
run_metadata['pretrained_weights_sha256'] = CONFIG['weights_sha256']
write_json(OUTPUT_ROOT / 'run-metadata.json', run_metadata)
print(json.dumps(run_metadata, indent=2, sort_keys=True))

## Construct train and validation data only

`fully_synthetic` is target `1`; `camera_origin` is target `0`. The held-out `test` split is intentionally never opened by this notebook. It remains unavailable for later benchmark reporting. Training augmentation applies only to the train split; validation uses the ImageNet preprocessing associated with the pinned pretrained weights.

In [ ]:
from PIL import Image
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import MobileNet_V3_Small_Weights, mobilenet_v3_small

weights = MobileNet_V3_Small_Weights.IMAGENET1K_V1
weight_transforms = weights.transforms()
train_transform = transforms.Compose([
    transforms.RandomResizedCrop(CONFIG['image_size'], scale=(0.8, 1.0), antialias=True),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.ToTensor(),
    transforms.Normalize(mean=weight_transforms.mean, std=weight_transforms.std),
])
validation_transform = weight_transforms

class PortraitDataset(Dataset):
    def __init__(self, records, root, transform):
        self.records = records
        self.root = root
        self.transform = transform

    def __len__(self):
        return len(self.records)

    def __getitem__(self, index):
        record = self.records[index]
        with Image.open(self.root / record['relative_path']) as image:
            image = image.convert('RGB')
        return self.transform(image), float(LABEL_TO_TARGET[record['label']])

records = record_document['records']
train_records = [record for record in records if record['split'] == 'train']
validation_records = [record for record in records if record['split'] == 'validation']
if not train_records or not validation_records:
    raise ValueError('The private manifest must include both train and validation records.')

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % (2 ** 32)
    random.seed(worker_seed)
    np.random.seed(worker_seed)

loader_generator = torch.Generator().manual_seed(CONFIG['seed'])
train_loader = DataLoader(
    PortraitDataset(train_records, CONFIG['data_root'], train_transform),
    batch_size=CONFIG['batch_size'], shuffle=True, num_workers=CONFIG['num_workers'],
    worker_init_fn=seed_worker, generator=loader_generator, pin_memory=torch.cuda.is_available(),
)
validation_loader = DataLoader(
    PortraitDataset(validation_records, CONFIG['data_root'], validation_transform),
    batch_size=CONFIG['batch_size'], shuffle=False, num_workers=CONFIG['num_workers'],
    worker_init_fn=seed_worker, pin_memory=torch.cuda.is_available(),
)

## Fine-tune the lightweight pretrained backbone

The attached checkpoint is hashed before loading; the model does not download weights at runtime. Only the final classifier is initialized for the two-class task. The validation AUROC is a model-selection metric, not a calibrated probability or operating threshold.

In [ ]:
from torch import nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = mobilenet_v3_small(weights=None)
checkpoint_state = torch.load(CONFIG['weights_path'], map_location='cpu', weights_only=True)
model.load_state_dict(checkpoint_state)
model.classifier[3] = nn.Linear(model.classifier[3].in_features, 1)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.AdamW(
    model.parameters(), lr=CONFIG['learning_rate'], weight_decay=CONFIG['weight_decay']
)

@torch.no_grad()
def validate():
    model.eval()
    scores, targets, losses = [], [], []
    for images, labels in validation_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)
        logits = model(images)
        losses.append(criterion(logits, labels).item())
        scores.extend(torch.sigmoid(logits).squeeze(1).cpu().tolist())
        targets.extend(labels.squeeze(1).cpu().int().tolist())
    return {'loss': float(np.mean(losses)), 'auroc': binary_auroc(scores, targets)}


In [ ]:
history = []
best_auroc = float('-inf')
best_epoch = None
checkpoint_path = OUTPUT_ROOT / 'mobilenetv3-small-state-dict.pt'

for epoch in range(1, CONFIG['epochs'] + 1):
    model.train()
    train_losses = []
    for images, labels in train_loader:
        images = images.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True).unsqueeze(1)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), labels)
        loss.backward()
        optimizer.step()
        train_losses.append(loss.item())

    validation = validate()
    epoch_result = {
        'epoch': epoch,
        'train_loss': float(np.mean(train_losses)),
        'validation_loss': validation['loss'],
        'validation_auroc': validation['auroc'],
    }
    history.append(epoch_result)
    if validation['auroc'] > best_auroc:
        best_auroc = validation['auroc']
        best_epoch = epoch
        torch.save({'model_state_dict': model.state_dict(), 'epoch': epoch}, checkpoint_path)
    print(epoch_result)

write_json(OUTPUT_ROOT / 'training-history.json', {'epochs': history})
run_metadata['selected_epoch'] = best_epoch
run_metadata['validation_model_selection_auroc'] = best_auroc
run_metadata['checkpoint_sha256'] = sha256_file(checkpoint_path)
write_json(OUTPUT_ROOT / 'run-metadata.json', run_metadata)
print(f'Private checkpoint: {checkpoint_path}')
print('Do not treat validation AUROC as an operating threshold or publish this artifact before the later evaluation and model-card milestones.')

## Preserve the run, keep it private, and stop

The output directory contains a state dictionary, `run-metadata.json`, and `training-history.json`. Keep all Kaggle outputs private. The next milestone exports a selected model to ONNX with a model card, licence, checksums, and a separately reproducible threshold configuration. Benchmarking later consumes the held-out test split. Do not use this notebook’s validation score as evidence that any individual portrait is authentic or synthetic.